# Load imports

In [1]:
import pandas as pd
import geopandas as gpd

# GeoJSON Files (Loaded with GeoPandas)
GeoJSON files contain geometry (points, lines, or polygons).
These datasets must be loaded with GeoPandas because I need their shapes for:

Spatial joins

Mapping

Neighborhood aggregation

Calculating zoning intensity by area

### 1. Zoning (GeoJSON)
Contains polygon geometries for zoning districts.

Required for spatial joins with parcels and subareas.

Used to calculate redevelopment potential.

### 2. Parcels (GeoJSON)
Contains parcel boundary polygons.

Needed to attach zoning to parcels and neighborhoods.

Supports parcel‑level spatial analysis.

### 3. Community Planning Areas / Subareas (GeoJSON)
Contains neighborhood boundary polygons.

Used as the geographic unit for aggregating all metrics.

Essential for spatial joins with zoning and permits.

In [2]:
# Load spatial datasets
zoning = gpd.read_file("../data/Zoning_Vw_-3635171973256313343 (1).geojson")

In [3]:
parcels = gpd.read_file("../data/Parcels_view_-4743305439210361335.geojson")

In [4]:
subareas = gpd.read_file("../data/Community_Planning_Areas_(Subareas)_.geojson")

# CSV Files (Loaded with pandas → Converted to GeoPandas)
CSV files contain tabular attributes, but in this project the permit datasets also include latitude and longitude.
We load them with pandas first, then convert them into GeoDataFrames so we can:

Map permit locations

Join permits to neighborhoods

Calculate permit density

Integrate them into the Neighborhood Transformation Index

### 4. Building Permits (CSV)
Stored as a table with permit details and lat/lng.

Converted into point geometry for spatial joins.

CSV version is cleaner and faster to load than the GeoJSON version.

### 5. Permit Applications (CSV)
Same structure as issued permits.

Includes lat/lng for spatial conversion.

Used to measure pipeline development activity.

In [5]:
# Load tabular datasets
permits = pd.read_csv("../data/Building_Permits_Issued_2_-3206176320106085339.csv")
applications = pd.read_csv("../data/Building_Permit_Applications_Feature_Layer_view_-3134924856831433637.csv")

# Inspecting the data

In [6]:
zoning.head()

,OBJECTID,ORD_DATE,CASE_NO,ORDINANCE,ZONE_DESC,NAME,GlobalID,geometry
0,1,"Tue, 24 Dec 1974 06:00:00 GMT",None,O73-650,CS,None,9c132ca3-3efc-4094-9825-bf3c935267ab,"POLYGON ((-86.87487 36.28165, -86.87468 36.281..."
1,2,"Thu, 01 Jan 1998 06:00:00 GMT",,O96-555,CS,None,18741a88-dae7-4fa0-b13a-a246e8fc3b04,"POLYGON ((-86.80117 36.27836, -86.80038 36.277..."
2,3,"Tue, 24 Dec 1974 06:00:00 GMT",,O73-650,R15,None,7b22f667-a035-4e47-a3d2-5aa0bbd72241,"POLYGON ((-86.80425 36.28038, -86.80438 36.279..."
3,4,"Thu, 28 May 2009 05:00:00 GMT",2009Z-012PR-001,BL2009-424,AR2A,None,758fcde4-6aac-47bc-b3dc-8504bd5eae73,"POLYGON ((-86.77879 36.29175, -86.77882 36.291..."
4,5,"Sat, 24 May 2003 05:00:00 GMT",2003Z-020G-02,BL2003-1393,R20,None,1641774a-7384-4a3d-9212-bf52426608db,"POLYGON ((-86.79217 36.27773, -86.79228 36.277..."


In [7]:
parcels.head()

,OBJECTID,STANPAR,FEATURETYPE,FLOORNUMBER,ParID,Tract,Council,TaxDist,Owner,OwnDate,...,IsRegular,LUCode,LUDesc,LandAppr,ImprAppr,TotlAppr,Zoning,Lat,Lon,geometry
0,1,110010A00200CO,Condo,None,455505.0,37015610,12,GSD,"BELEW, STEPHEN JOSEPH","Fri, 08 Sep 2023 05:00:00 GMT",...,N,015,RESIDENTIAL CONDO,95200.0,376400.0,471600.0,SP,36.148727,-86.580355,"POLYGON ((-86.58036 36.14877, -86.58035 36.148..."
1,2,083010O90000CO,Common Area,None,455507.0,37011700,05,USD,O.I.C. TENFIFTYONE PETWAY AVENUE,"Thu, 22 Jul 2021 05:00:00 GMT",...,N,010,VACANT RESIDENTIAL LAND,100.0,0.0,100.0,RS5,36.187147,-86.747849,"POLYGON ((-86.74769 36.18747, -86.74778 36.186..."
2,3,114040A06500CO,Condo,None,455508.0,37018301,35,GSD,"MCCLELLAND, JASON & DANIELLE","Tue, 05 Dec 2023 06:00:00 GMT",...,N,010,VACANT RESIENTIAL LAND,90000.0,346000.0,436000.0,SP,36.120900,-86.920964,"POLYGON ((-86.92101 36.12094, -86.92090 36.120..."
3,4,114040A06300CO,Condo,None,455509.0,37018301,35,GSD,"HO, JUI-LIEN CHOU","Fri, 08 Dec 2023 06:00:00 GMT",...,N,010,VACANT RESIDENTIAL LAND,90000.0,346000.0,436000.0,SP,36.120855,-86.920987,"POLYGON ((-86.92103 36.12090, -86.92092 36.120..."
4,5,114040A06400CO,Condo,None,455512.0,37018301,35,GSD,"SILVERMAN, GREG","Thu, 05 Sep 2024 05:00:00 GMT",...,N,010,VACANT RESIDENTIAL LAND,90000.0,349500.0,439500.0,SP,36.120886,-86.921480,"POLYGON ((-86.92140 36.12089, -86.92142 36.120..."


In [8]:
subareas.head()

,OBJECTID,CommunityNumber,CommunityName,GlobalID,geometry
0,1,2,Parkwood - Union Hill,baa3fada-cf8c-4384-899e-b296a3f0d634,"POLYGON ((-86.75375 36.40431, -86.75362 36.404..."
1,2,1,Joelton,57192bac-4b35-433e-9a57-0eb4e23f8ca0,"POLYGON ((-86.90741 36.39053, -86.90728 36.390..."
2,3,4,Madison,68911bc4-caf0-4d28-bc9f-c64c27a530fb,"POLYGON ((-86.66885 36.30088, -86.67049 36.301..."
3,4,3,Bordeaux - Whites Creek - Haynes Trinity,1112fbc3-285c-4c29-b548-11223136195e,"POLYGON ((-86.81409 36.30151, -86.81326 36.300..."
4,5,14,Donelson - Hermitage - Old Hickory,4c9ba617-436b-45be-8d80-0740d86a4809,"POLYGON ((-86.58100 36.20931, -86.58109 36.208..."


In [9]:
permits.head()

,Permit #,Permit Type Description,Permit Subtype Description,Parcel,Date Entered,Date Issued,Construction Cost,Address,City,State,...,IVR Tracking,Purpose,Council District,Census Tract,Longitude,Latitude,ObjectId,Zip Code,x,y
0,2019067389,Building Use & Occupancy,"Multifamily, Tri-Plex, Quad, Apartments",05100017300,11/4/2019 6:00:00 AM,4/4/2023 5:00:00 AM,1,600 CREATIVE WAY,MADISON,TN,...,3728777,MASTER PERMIT ONLY NO CONSTRUCTION THIS PERMIT...,5.0,37010802,-86.740236,36.243639,1,37115,-9.655879e+06,4.334198e+06
1,2020006147,Building Residential - New,Single Family Residence,00800026100,1/28/2020 6:00:00 AM,4/21/2023 5:00:00 AM,178132,4168 BAXTER RD,JOELTON,TN,...,3754519,To construct a new single family residence wit...,1.0,37010103,-86.907892,36.364056,2,37080,-9.674542e+06,4.350831e+06
2,2020008205,Building Residential - New,Single Family Residence,05308013800,2/6/2020 6:00:00 AM,7/19/2023 5:00:00 AM,734113,410 30TH ST,OLD HICKORY,TN,...,3757502,12/28/2020 Applicant is submitting revised ele...,11.0,37010502,-86.635039,36.248312,3,37138,-9.644168e+06,4.334843e+06
3,2020037584,Building Commercial - Rehab,Day Care Center (Up To 75) - Child Care,05106006200,6/17/2020 5:00:00 AM,4/9/2024 5:00:00 AM,82455,1022 S GRAYCROFT AVE,MADISON,TN,...,3837660,parcel is zoned RM9 and RS20 and contains 11.1...,3.0,37010802,-86.736233,36.252179,4,37115,-9.655433e+06,4.335376e+06
4,2020050957,Building Residential - New,"Accessory Structure, Pools - Residential",11407002600,8/17/2020 5:00:00 AM,4/20/2023 5:00:00 AM,75000,7545 B OAKHAVEN TRCE,NASHVILLE,TN,...,3862490,To install an inground 18x36 sqaure shaped poo...,35.0,37018301,-86.928892,36.118691,5,37209,-9.676880e+06,4.316965e+06


In [10]:
applications.head()

,Permit #,Permit Type Description,Permit Subtype Description,Parcel,Date Entered,Date Issued,Construction Cost,Address,City,State,...,Permit SubType,IVR Tracking,Purpose,Council District,Lon,Lat,ObjectId,Zip Code,x,y
0,D2020047607,Building Use & Occupancy,"Accessory Structure, Pools - Residential",11704002200,10/24/2023 5:00:00 AM,NaN,100000,2509 BELMONT BLVD,NASHVILLE,TN,...,CAA14U017,3856896,Proposed 13' x 32' Swimming Pool in rear yard.,18.0,-86.794539,36.125197,1,37212,-9.661924e+06,4.317862e+06
1,D2021026275,Building Residential - Addition,Single Family Residence,16109004000,8/2/2024 5:00:00 AM,NaN,318761,5500 THALMAN DR,BRENTWOOD,TN,...,CAA01R301,3974584,5500 Thalman Drive,26.0,-86.749685,36.045886,2,37027,-9.656931e+06,4.306937e+06
2,D2023014026,Building Residential - Addition,Single Family Residence,12912005600,3/3/2023 6:00:00 AM,NaN,35984,1001 PERCY WARNER BLVD,NASHVILLE,TN,...,CAA01R301,4279927,To construct a 744 sq ft carport addition to t...,23.0,-86.881114,36.090131,3,37205,-9.671561e+06,4.313030e+06
3,D2023027056,Building Residential - New,"Multifamily, Tri-Plex, Quad, Apartments",09203002900,4/25/2023 5:00:00 AM,NaN,0,2405 E JEFFERSON ST,NASHVILLE,TN,...,CAA03R398,4311444,Permit not required by zoning The work for ...,21.0,-86.813076,36.168483,4,37208,-9.663987e+06,4.323829e+06
4,D2023029429,Building Demolition Permit,Demolition Permit - Residential,06108020400,12/17/2024 6:00:00 AM,NaN,20000,1130 WINDING WAY,NASHVILLE,TN,...,CAZ01A001,4318467,Demo existing home in need of much repairs sit...,7.0,-86.721786,36.226283,5,37216,-9.653825e+06,4.331802e+06


In [11]:
zoning.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   OBJECTID   5937 non-null   int64   
 1   ORD_DATE   5922 non-null   object  
 2   CASE_NO    5829 non-null   object  
 3   ORDINANCE  5925 non-null   object  
 4   ZONE_DESC  5937 non-null   object  
 5   NAME       1015 non-null   object  
 6   GlobalID   5937 non-null   object  
 7   geometry   5932 non-null   geometry
dtypes: geometry(1), int64(1), object(6)
memory usage: 371.2+ KB


In [12]:
parcels.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 285934 entries, 0 to 285933
Data columns (total 43 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   OBJECTID      285934 non-null  int64   
 1   STANPAR       285934 non-null  object  
 2   FEATURETYPE   285934 non-null  object  
 3   FLOORNUMBER   21670 non-null   object  
 4   ParID         285884 non-null  float64 
 5   Tract         285883 non-null  object  
 6   Council       285882 non-null  object  
 7   TaxDist       282812 non-null  object  
 8   Owner         285884 non-null  object  
 9   OwnDate       285883 non-null  object  
 10  SalePrice     227494 non-null  float64 
 11  OwnInstr      285883 non-null  object  
 12  OwnAddr1      285385 non-null  object  
 13  OwnAddr2      75834 non-null   object  
 14  OwnAddr3      75748 non-null   object  
 15  OwnCity       285384 non-null  object  
 16  OwnState      285634 non-null  object  
 17  OwnCountry    285633 

In [13]:
subareas.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   OBJECTID         14 non-null     int64   
 1   CommunityNumber  14 non-null     int64   
 2   CommunityName    14 non-null     object  
 3   GlobalID         14 non-null     object  
 4   geometry         14 non-null     geometry
dtypes: geometry(1), int64(2), object(2)
memory usage: 688.0+ bytes


In [14]:
permits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29619 entries, 0 to 29618
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Permit #                    29619 non-null  object 
 1   Permit Type Description     29619 non-null  object 
 2   Permit Subtype Description  29619 non-null  object 
 3   Parcel                      29619 non-null  object 
 4   Date Entered                29619 non-null  object 
 5   Date Issued                 29619 non-null  object 
 6   Construction Cost           29619 non-null  int64  
 7   Address                     29619 non-null  object 
 8   City                        29617 non-null  object 
 9   State                       29619 non-null  object 
 10  Subdivision Lot             29619 non-null  object 
 11  Contact                     29619 non-null  object 
 12  Permit Type                 29619 non-null  object 
 13  Permit SubType              296

In [15]:
applications.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5645 entries, 0 to 5644
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Permit #                    5645 non-null   object 
 1   Permit Type Description     5645 non-null   object 
 2   Permit Subtype Description  5645 non-null   object 
 3   Parcel                      5645 non-null   object 
 4   Date Entered                5645 non-null   object 
 5   Date Issued                 0 non-null      float64
 6   Construction Cost           5645 non-null   int64  
 7   Address                     5645 non-null   object 
 8   City                        5644 non-null   object 
 9   State                       5645 non-null   object 
 10  Subdivision Lot             5645 non-null   object 
 11  Contact                     5645 non-null   object 
 12  Permit Type                 5645 non-null   object 
 13  Permit SubType              5645 

# Cleaning up the data!

# ZONING CLEANING

### Step 1 — Load and inspect the zoning dataset  
I’m keeping only the zoning fields I need and standardizing the zoning descriptions.

In [16]:
# Keep only relevant zoning fields
zoning_clean = zoning_clean = zoning[[
    "OBJECTID", "ORD_DATE", "CASE_NO", "ORDINANCE",
    "ZONE_DESC", "NAME", "geometry"
]].copy()

# Standardize zoning descriptions
zoning_clean["ZONE_DESC"] = zoning_clean["ZONE_DESC"].astype(str).str.strip().str.upper()

# Drop rows without geometry
zoning_clean = zoning_clean.dropna(subset=["geometry"])

zoning_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 5932 entries, 0 to 5936
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   OBJECTID   5932 non-null   int64   
 1   ORD_DATE   5917 non-null   object  
 2   CASE_NO    5825 non-null   object  
 3   ORDINANCE  5920 non-null   object  
 4   ZONE_DESC  5932 non-null   object  
 5   NAME       1014 non-null   object  
 6   geometry   5932 non-null   geometry
dtypes: geometry(1), int64(1), object(5)
memory usage: 370.8+ KB


### Step 2 — Convert zoning dates
I’m converting the ordinance date into a proper datetime field.

In [17]:
zoning_clean["ORD_DATE"] = pd.to_datetime(zoning_clean["ORD_DATE"], errors="coerce")

### Step 3 — Confirm zoning CRS
I’m making sure zoning uses EPSG:4326 (the standard identifier for the WGS 84 coordinate reference system {thanks google!}) so it aligns with the rest of the project.

In [18]:
if zoning_clean.crs != "EPSG:4326":
    zoning_clean = zoning_clean.to_crs("EPSG:4326")

# PARCELS CLEANING

### Step 4 — Decide which parcel fields to keep  
I’m keeping the parcel fields that allow me to calculate sale price trends by neighborhood. These include sale price, ownership date, parcel ID, zoning, acreage, and geometry. Everything else can be dropped.

In [19]:
parcels_clean = parcels[[
    "ParID",
    "SalePrice",
    "OwnDate",
    "PropDate",
    "PropZip",
    "Zoning",
    "Acres",
    "geometry"
]].copy()

# Convert dates
parcels_clean["OwnDate"] = pd.to_datetime(parcels_clean["OwnDate"], errors="coerce")
parcels_clean["PropDate"] = pd.to_datetime(parcels_clean["PropDate"], errors="coerce")

# Standardize zoning
parcels_clean["Zoning"] = parcels_clean["Zoning"].astype(str).str.strip().str.upper()

# Drop rows without geometry
parcels_clean = parcels_clean.dropna(subset=["geometry"])

parcels_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 285934 entries, 0 to 285933
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ParID      285884 non-null  float64       
 1   SalePrice  227494 non-null  float64       
 2   OwnDate    285883 non-null  datetime64[ns]
 3   PropDate   285882 non-null  datetime64[ns]
 4   PropZip    285884 non-null  object        
 5   Zoning     285934 non-null  object        
 6   Acres      285884 non-null  float64       
 7   geometry   285934 non-null  geometry      
dtypes: datetime64[ns](2), float64(3), geometry(1), object(2)
memory usage: 17.5+ MB


### Step 5 — Confirm parcels CRS
I’m making sure parcels use the same CRS as zoning.

In [20]:
if parcels_clean.crs != "EPSG:4326":
    parcels_clean = parcels_clean.to_crs("EPSG:4326")

# SUBAREAS CLEANING
### Step 6 — Clean subareas
I’m keeping the subarea name and geometry and confirming CRS.

In [21]:
subareas_clean = subareas[["CommunityNumber", "CommunityName", "geometry"]].copy()

if subareas_clean.crs != "EPSG:4326":
    subareas_clean = subareas_clean.to_crs("EPSG:4326")

subareas_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   CommunityNumber  14 non-null     int64   
 1   CommunityName    14 non-null     object  
 2   geometry         14 non-null     geometry
dtypes: geometry(1), int64(1), object(1)
memory usage: 464.0+ bytes


# PERMITS CLEANING
### Step 7 — Clean permit fields
I’m keeping the fields I need and converting dates.

In [22]:
permits_clean = permits[[
    "Permit #", "Permit Type Description", "Permit Subtype Description",
    "Parcel", "Date Entered", "Date Issued", "Construction Cost",
    "Longitude", "Latitude"
]].copy()

# Convert dates
permits_clean["Date Entered"] = pd.to_datetime(permits_clean["Date Entered"], errors="coerce")
permits_clean["Date Issued"] = pd.to_datetime(permits_clean["Date Issued"], errors="coerce")

permits_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29619 entries, 0 to 29618
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Permit #                    29619 non-null  object        
 1   Permit Type Description     29619 non-null  object        
 2   Permit Subtype Description  29619 non-null  object        
 3   Parcel                      29619 non-null  object        
 4   Date Entered                29619 non-null  datetime64[ns]
 5   Date Issued                 29619 non-null  datetime64[ns]
 6   Construction Cost           29619 non-null  int64         
 7   Longitude                   29619 non-null  float64       
 8   Latitude                    29619 non-null  float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(4)
memory usage: 2.0+ MB


### Step 8 — Convert permits to point geometry
I’m converting the lat/lng columns into point geometry.

In [23]:
permits_gdf = gpd.GeoDataFrame(
    permits_clean,
    geometry=gpd.points_from_xy(permits_clean.Longitude, permits_clean.Latitude),
    crs="EPSG:4326"
)

In [24]:
permits_gdf.head()

,Permit #,Permit Type Description,Permit Subtype Description,Parcel,Date Entered,Date Issued,Construction Cost,Longitude,Latitude,geometry
0,2019067389,Building Use & Occupancy,"Multifamily, Tri-Plex, Quad, Apartments",05100017300,2019-11-04 06:00:00,2023-04-04 05:00:00,1,-86.740236,36.243639,POINT (-86.74024 36.24364)
1,2020006147,Building Residential - New,Single Family Residence,00800026100,2020-01-28 06:00:00,2023-04-21 05:00:00,178132,-86.907892,36.364056,POINT (-86.90789 36.36406)
2,2020008205,Building Residential - New,Single Family Residence,05308013800,2020-02-06 06:00:00,2023-07-19 05:00:00,734113,-86.635039,36.248312,POINT (-86.63504 36.24831)
3,2020037584,Building Commercial - Rehab,Day Care Center (Up To 75) - Child Care,05106006200,2020-06-17 05:00:00,2024-04-09 05:00:00,82455,-86.736233,36.252179,POINT (-86.73623 36.25218)
4,2020050957,Building Residential - New,"Accessory Structure, Pools - Residential",11407002600,2020-08-17 05:00:00,2023-04-20 05:00:00,75000,-86.928892,36.118691,POINT (-86.92889 36.11869)


# PERMIT APPLICATIONS CLEANING
### Step 9 — Clean application fields
I’m keeping the fields I need and converting dates.

In [25]:
applications_clean = applications[[
    "Permit #", "Permit Type Description", "Permit Subtype Description",
    "Parcel", "Date Entered", "Construction Cost",
    "Lon", "Lat"
]].copy()

applications_clean["Date Entered"] = pd.to_datetime(applications_clean["Date Entered"], errors="coerce")

applications_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5645 entries, 0 to 5644
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Permit #                    5645 non-null   object        
 1   Permit Type Description     5645 non-null   object        
 2   Permit Subtype Description  5645 non-null   object        
 3   Parcel                      5645 non-null   object        
 4   Date Entered                5645 non-null   datetime64[ns]
 5   Construction Cost           5645 non-null   int64         
 6   Lon                         5645 non-null   float64       
 7   Lat                         5645 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 352.9+ KB


### Step 10 — Convert applications to point geometry
I’m converting the lat/lng columns into point geometry.

In [26]:
applications_gdf = gpd.GeoDataFrame(
    applications_clean,
    geometry=gpd.points_from_xy(applications_clean.Lon, applications_clean.Lat),
    crs="EPSG:4326"
)

applications_gdf.head()

,Permit #,Permit Type Description,Permit Subtype Description,Parcel,Date Entered,Construction Cost,Lon,Lat,geometry
0,D2020047607,Building Use & Occupancy,"Accessory Structure, Pools - Residential",11704002200,2023-10-24 05:00:00,100000,-86.794539,36.125197,POINT (-86.79454 36.12520)
1,D2021026275,Building Residential - Addition,Single Family Residence,16109004000,2024-08-02 05:00:00,318761,-86.749685,36.045886,POINT (-86.74968 36.04589)
2,D2023014026,Building Residential - Addition,Single Family Residence,12912005600,2023-03-03 06:00:00,35984,-86.881114,36.090131,POINT (-86.88111 36.09013)
3,D2023027056,Building Residential - New,"Multifamily, Tri-Plex, Quad, Apartments",09203002900,2023-04-25 05:00:00,0,-86.813076,36.168483,POINT (-86.81308 36.16848)
4,D2023029429,Building Demolition Permit,Demolition Permit - Residential,06108020400,2024-12-17 06:00:00,20000,-86.721786,36.226283,POINT (-86.72179 36.22628)
